<a href="https://colab.research.google.com/github/FANGxPC/LIFELOG_AI/blob/master/LIFELOG_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install specialized versions for Qwen2-VL video support
!pip install git+https://github.com/huggingface/transformers@21fac7abba2a37fae86106f87fcf9974fd1e3830 accelerate -q
!pip install qwen-vl-utils[decord] av -q
!pip install faster-whisper -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import torch
import base64
from faster_whisper import WhisperModel

import time
from IPython.display import Javascript, display
from google.colab.output import eval_js
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# Force torch to be global
global torch

print("🚀 Initializing Qwen2-VL...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct", torch_dtype="auto", device_map="auto"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
whisper_model = WhisperModel("base.en", device="cuda", compute_type="float16")

🚀 Initializing Qwen2-VL...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
def capture_10s_moment(filename='memory.webm'):
    js_code = '''
    (async function() {
      const stream = await navigator.mediaDevices.getUserMedia({video: true, audio: true});
      const recorder = new MediaRecorder(stream);
      const chunks = [];
      recorder.ondataavailable = (e) => chunks.push(e.data);
      recorder.start();
      console.log("Recording 10s...");
      await new Promise(r => setTimeout(r, 10000));
      recorder.stop();
      return new Promise(r => {
        recorder.onstop = () => {
          const blob = new Blob(chunks, {type: 'video/webm'});
          const reader = new FileReader();
          reader.readAsDataURL(blob);
          reader.onloadend = () => r(reader.result);
          stream.getTracks().forEach(t => t.stop());
        };
      });
    })()
    '''
    data = eval_js(js_code)
    binary = base64.b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

In [5]:
def sync_10s_moment():
    import torch
    try:
        # 1. Capture 10s of life
        video_path = capture_10s_moment()

        # 2. Transcription
        print("🎙️ Transcribing...")
        segments, _ = whisper_model.transcribe(video_path)
        transcript = " ".join([s.text for s in segments]).strip()

        # 3. Reasoning - Improved Prompt & Video Handling
        print("🧠 Qwen2-VL analyzing video...")

        # We give the model a bit more "room to breathe" before forcing JSON
        messages = [{
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": video_path,
                    "fps": 1.0, # Increased sampling to 2 frames per second (20 frames total)
                },
                {
                    "type": "text",
                    "text": f"Audio heard: {transcript}\n\n"
                            "Task: You are a life-logging assistant. Watch this 10s video carefully. "
                            "First, describe what you see. Then, provide a JSON summary. "
                            "JSON format: {\"loc\": \"room type\", \"actions\": \"what happened\", \"objs\": [\"list\"]}"
                }
            ]
        }]

        # 4. Generate with specific JSON-friendly settings
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            # max_new_tokens increased to 300 to allow for the description + JSON
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=True, # Adding a bit of creativity helps avoid empty outputs
                temperature=0.7,
                top_p=0.9
            )
            output = processor.batch_decode(gen_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

        print("\n" + "="*45)
        print("✅ SYNC COMPLETE")
        print(f"🎙️ TRANSCRIPT: {transcript}")
        print(f"🧠 AI RESPONSE:\n{output}")
        print("="*45 + "\n")

    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        torch.cuda.empty_cache()

# Run it
sync_10s_moment()

🎙️ Transcribing...
🧠 Qwen2-VL analyzing video...

✅ SYNC COMPLETE
🎙️ TRANSCRIPT: Hello, hello.  Hello, hello.
🧠 AI RESPONSE:
```json
{
  "loc": "living room",
  "actions": "waving",
  "objs": ["hand"]
}
```



Checkign thigns


In [7]:
def sync_10s_moment():
    import torch
    try:
        # 1. Capture 10s of life
        video_path = capture_10s_moment()

        # 2. Transcription
        print("🎙️ Transcribing...")
        segments, _ = whisper_model.transcribe(video_path)
        transcript = " ".join([s.text for s in segments]).strip()

        # 3. Reasoning - Improved Prompt & Video Handling
        print("🧠 Qwen2-VL analyzing video...")

        # We give the model a bit more "room to breathe" before forcing JSON
        messages = [{
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": video_path,
                    "fps": 1.0, # Increased sampling to 2 frames per second (20 frames total)
                },
                {
                    "type": "text",
                    "text": f"Audio heard: {transcript}\n\n"
                            "Task: You are a life-logging assistant. Watch this 10s video carefully. "
                            "Describe every bit of thing you see and extract every info you can.Then anwser the question from Audio heard: {transcript}"


                }
            ]
        }]

        # 4. Generate with specific JSON-friendly settings
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            # max_new_tokens increased to 300 to allow for the description + JSON
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=True, # Adding a bit of creativity helps avoid empty outputs
                temperature=0.7,
                top_p=0.9
            )
            output = processor.batch_decode(gen_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

        print("\n" + "="*45)
        print("✅ SYNC COMPLETE")
        print(f"🎙️ TRANSCRIPT: {transcript}")
        print(f"🧠 AI RESPONSE:\n{output}")
        print("="*45 + "\n")

    except Exception as e:
        print(f"❌ Error: {e}")
    finally:
        torch.cuda.empty_cache()

# Run it
sync_10s_moment()

🎙️ Transcribing...
🧠 Qwen2-VL analyzing video...

✅ SYNC COMPLETE
🎙️ TRANSCRIPT: Tell me how many hands I have.
🧠 AI RESPONSE:
The person in the video is wearing a white t-shirt and has a beard. They are standing in a room with a bed and a wall in the background. The person is making hand gestures and speaking to the camera.

